In [ ]:
import torch
from src.data.puzzle_dataset import PuzzleDataset
from src.data.criterion.nonogram import NonogramLoss
from src.models.simple_nn import SimpleNeuralNetwork
from src.models.simple_transformer import SimpleTransformer
from src.models.recursive_mlp import RecursiveMLP
from src.training.reward_trainer import RewardTrainer
from src.data.criterion.nonogram import grid_to_row_clues
import matplotlib.pyplot as plt
import os

if "_NOTEBOOK_PARENT_DIR" not in globals():
    _NOTEBOOK_PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(_NOTEBOOK_PARENT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Database

In [ ]:
dataset_name = "nonogram_10x10"

match dataset_name:
    case "nonogram_5x5":
        target_shape = (5, 5)
        location = "processed/nonogram_single_sample_5x5.npy"
        clues = torch.tensor([[[[2,1,0], [2,1,0], [4,0,0], [0,0,0], [1,0,0]], [[1,0,0], [3,1,0], [1,1,0], [2,0,0], [1,1,0]]]], dtype=torch.float32).to(device)
        true_grid = torch.tensor([[0, 1, 1, 0, 1], [1, 1, 0, 1, 0], [0, 1, 1, 1, 1], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0]], dtype=torch.float32).to(device)
    case "nonogram_10x10":
        target_shape = (10, 10)
        location = "processed/nonogram_single_sample_10x10.npy"
        clues = torch.tensor([[[[3,1,1,0,0],[3,3,0,0,0],[2,1,1,1,0],[3,2,0,0,0],[1,0,0,0,0],[6,0,0,0,0],[1,6,0,0,0],[6,0,0,0,0],[3,1,0,0,0],[5,0,0,0,0]],[[3,1,0,0,0],[3,0,0,0,0],[2,0,0,0,0],[2,0,0,0,0],[5,0,0,0,0],[1,5,0,0,0],[1,5,0,0,0],[2,5,0,0,0],[2,1,3,1,0],[3,5,0,0,0]]]],dtype=torch.float32).to(device)
        true_grid = torch.tensor([[1, 1, 1, 0, 0, 0, 1, 0, 1, 0],[1, 1, 1, 0, 0, 0, 0, 1, 1, 1],[1, 1, 0, 1, 0, 0, 0, 1, 0, 1],[0, 0, 0, 1, 1, 1, 0, 0, 1, 1],[0, 0, 0, 0, 1, 0, 0, 0, 0, 0],[0, 0, 0, 0, 1, 1, 1, 1, 1, 1],[1, 0, 0, 0, 1, 1, 1, 1, 1, 1],[0, 0, 0, 0, 1, 1, 1, 1, 1, 1],[0, 0, 0, 0, 0, 1, 1, 1, 0, 1],[0, 0, 0, 0, 0, 1, 1, 1, 1, 1]], dtype=torch.float32).to(device)
    case "nonogram_15x15":
        target_shape = (15, 15)
        location = "processed/nonogram_single_sample_15x15.npy"
        clues = torch.tensor([[[[2, 7, 0, 0, 0, 0, 0, 0],[9, 0, 0, 0, 0, 0, 0, 0],[8, 0, 0, 0, 0, 0, 0, 0],[1, 1, 1, 1, 1, 0, 0, 0],[3, 3, 0, 0, 0, 0, 0, 0],[1, 2, 2, 0, 0, 0, 0, 0],[2, 3, 3, 0, 0, 0, 0, 0],[2, 3, 0, 0, 0, 0, 0, 0],[3, 0, 0, 0, 0, 0, 0, 0],[1, 3, 0, 0, 0, 0, 0, 0],[2, 1, 3, 4, 0, 0, 0, 0],[2, 9, 0, 0, 0, 0, 0, 0],[2, 9, 0, 0, 0, 0, 0, 0],[7, 1, 1, 0, 0, 0, 0, 0],[5, 5, 0, 0, 0, 0, 0, 0]],[[3, 2, 5, 0, 0, 0, 0, 0],[4, 1, 5, 0, 0, 0, 0, 0],[2, 2, 0, 0, 0, 0, 0, 0],[4, 2, 2, 0, 0, 0, 0, 0],[3, 2, 0, 0, 0, 0, 0, 0],[5, 1, 4, 0, 0, 0, 0, 0],[3, 3, 4, 0, 0, 0, 0, 0],[8, 3, 0, 0, 0, 0, 0, 0],[2, 1, 3, 0, 0, 0, 0, 0],[1, 2, 0, 0, 0, 0, 0, 0],[4, 0, 0, 0, 0, 0, 0, 0],[3, 1, 0, 0, 0, 0, 0, 0],[1, 7, 1, 0, 0, 0, 0, 0],[9, 1, 0, 0, 0, 0, 0, 0],[8, 1, 0, 0, 0, 0, 0, 0]]]], dtype=torch.float32).to(device)
        true_grid = torch.tensor([[1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],[0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1],[0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1],[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1],[1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1],[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1],[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],[1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],[1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],[1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],[1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0],[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]], dtype=torch.float32)

clues_flat = clues.flatten(start_dim=1)
dataset = PuzzleDataset(
    location, 
    flat=True, 
    target_shape=target_shape)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, 
    [train_size, test_size],
    generator=generator
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False
)


## Visualisation

In [ ]:
def visualise_matrix(matrix):
    inv_matrix = 1 - matrix
    fig, ax = plt.subplots()
    ax.imshow(inv_matrix.cpu().numpy(), cmap='gray')

    # Compute clues
    grid = matrix.reshape(1, *matrix.shape)
    row_clues = grid_to_row_clues(grid, K=3)
    col_clues = grid_to_row_clues(grid.transpose(1, 2), K=3)

    n_rows, n_cols = matrix.shape

    # Row clues on the left
    for r in range(n_rows):
        clue = row_clues[0, r].cpu().tolist()
        clue_str = "\n".join(f"{num:.2f}" for num in clue if num > 0) or "0.00"
        ax.text(-0.6, r, clue_str, va='center', ha='right', fontsize=9)

    # Column clues on the top
    for c in range(n_cols):
        clue = col_clues[0, c].cpu().tolist()
        clue_str = "\n".join(f"{num:.2f}" for num in clue if num > 0) or "0.00"
        ax.text(c, -0.6, clue_str, va='bottom', ha='center', fontsize=9)

    ax.set_xlim(-0.5, n_cols)
    ax.set_ylim(n_rows, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)
    grid_box = plt.Rectangle((-0.5, -0.5), n_cols, n_rows,
                             linewidth=1, edgecolor='black', facecolor='none')
    ax.add_patch(grid_box)

    plt.tight_layout()
    plt.show()

def visualise_guess(model, clues, criterion):
    model.eval()
    with torch.no_grad():
        output = model(clues)
        loss, _, _ = criterion(output, clues)
        visualise_matrix(output.reshape(target_shape))
        print(f"Loss: {loss.item():.4f}")
    model.train()

## Running

In [ ]:
def run_training(model_type="nn", total_epochs=5, visualise_every=1):
    torch.manual_seed(42)
    match model_type:
        case "nn":
            # 10x10: 256 hidden size, 9 layers, 0.3 dropout
            model = SimpleNeuralNetwork(
                hidden_size=256,
                num_layers=9,
                dropout=0.3,
                input_size=dataset.input_shape,
                output_size=dataset.target_shape).to(device)
        case "transformer":
            model = SimpleTransformer(
                hidden_size=256,
                num_layers=9,
                dropout=0.3,
                input_size=dataset.input_shape,
                output_size=dataset.target_shape).to(device)
        case "recursive_mlp":
            model = RecursiveMLP(
                hidden_size=2048,
                num_layers=4,
                dropout=0.3,
                input_size=dataset.input_shape,
                output_size=dataset.target_shape).to(device)
        case _:
            model = None

    criterion = NonogramLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-6)
    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-5)

    trainer = RewardTrainer(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        scheduler=scheduler,
        epochs=visualise_every
    )

    visualise_guess(model, clues_flat, criterion)
    for i in range(total_epochs//visualise_every):
        trainer.train()
        print("="*40)
        print(f"Epoch {i+1}:")
        visualise_guess(model, clues_flat, criterion)
        
    return model

In [ ]:
final_model = run_training(model_type="recursive_mlp", total_epochs=400, visualise_every=100)
visualise_matrix(true_grid)

In [ ]:
# Only works for recursive MLP, as we can change the number of layers at inference time
criterion = NonogramLoss().to(device)
for n in range(1, final_model.num_layers + 1):
    final_model.num_layers = n
    visualise_guess(final_model, clues_flat, criterion)